# TensorFlow BBBP Baseline

This notebook mirrors the BBBP baseline in TensorFlow/Keras using a character-level text representation of SMILES.

Task: binary classification on BBBP
Model: TextVectorization plus dense classifier

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
bbbp = pd.read_csv(DATA_DIR / 'BBBP.csv')
display(bbbp.head())

In [ ]:
X = bbbp['smiles']
y = bbbp['p_np']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

In [ ]:
vectorizer = tf.keras.layers.TextVectorization(
    standardize=None,
    split='character',
    output_mode='multi_hot',
    max_tokens=128,
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train.to_numpy()).batch(64))

def make_dataset(features, labels, training: bool):
    ds = tf.data.Dataset.from_tensor_slices((features.to_numpy(), labels.to_numpy().astype('float32')))
    if training:
        ds = ds.shuffle(buffer_size=len(features), seed=RANDOM_SEED)
    return ds.batch(64).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, training=True)
valid_ds = make_dataset(X_valid, y_valid, training=False)
test_ds = make_dataset(X_test, y_test, training=False)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'), tf.keras.metrics.AUC(name='auc')],
)

history = model.fit(train_ds, validation_data=valid_ds, epochs=15, verbose=0)
history_df = pd.DataFrame(history.history)
display(history_df.tail().round(4))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df.index + 1, history_df['loss'], label='train_loss')
plt.plot(history_df.index + 1, history_df['val_loss'], label='valid_loss')
plt.title('TensorFlow BBBP Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
test_probs = model.predict(test_ds, verbose=0).ravel()
test_pred = (test_probs >= 0.5).astype(int)
test_metrics = pd.DataFrame([
    {
        'accuracy': accuracy_score(y_test, test_pred),
        'f1': f1_score(y_test, test_pred),
        'roc_auc': roc_auc_score(y_test, test_probs),
    }
])
display(test_metrics.round(4))